# Práctico 3 — Modelado

**Mentoría M09 — El impacto de la IA en la fuerza laboral**

Dataset: [diplodatos-ai-impact-workforce](https://github.com/carolinapepe/diplodatos-ai-impact-workforce)

---

## Qué hace esta notebook

El Práctico 1 dejó dos hipótesis planteadas y el Práctico 2 preparó los datos para responderlas, dejando además un catálogo de variables por rol y una serie de advertencias puntuales para el momento de modelar. Esta notebook retoma la **Hipótesis 2** — ¿es posible predecir la pérdida o creación de empleo a partir de las características de adopción de IA de una empresa? — y aplica sobre ella un enfoque mixto: clasificación supervisada como método principal, y clustering no supervisado como evidencia complementaria e independiente.

## Resumen de decisiones (para leer antes que el código)

1. **El target no es `saldo_neto_empleo`.** Tiene fiabilidad 0,33 (P2, §10.6.2): dos tercios de su varianza es ruido de medición, y al estar centrado en cero, ese ruido cae justo sobre el borde de decisión de un clasificador binario ingenuo. En su lugar modelamos `tasa_creacion` (fiabilidad 0,90) y `tasa_desplazamiento` (fiabilidad 0,79) como dos clasificadores binarios independientes, cada uno partido por su mediana.
2. **El clustering nunca ve los resultados.** Se arma únicamente con variables de rol *predictor* (estructura, adopción, operación, gobernanza, país). Los resultados (`tasa_creacion`, `tasa_desplazamiento`, `productivity_change_percent`, etc.) se calculan *después*, por clúster, solo para describir — nunca para agrupar.
3. **Unidad de análisis: la tabla por empresa**, no el panel. El panel exige `GroupKFold` por `company_id` y el P2 mostró que su dimensión temporal es, para este target, ruido blanco (§10.6.1) — no se gana señal real usando las 150.000 filas del panel en vez de las 10.000 empresas.
4. **`task_automation_rate` se reporta con y sin.** Está a 0,91 de correlación con la adopción y conceptualmente pegada al mecanismo de desplazamiento (P2, §11.2): la incluimos porque es la hipótesis a testear, pero mostramos el efecto de sacarla.
5. **El clustering es complementario, no un reemplazo.** Sirve para triangular: si los perfiles de alta adopción muestran, en promedio, mejor balance de creación/desplazamiento que los de baja adopción, tenemos dos métodos independientes apuntando en la misma dirección — evidencia más fuerte que un solo modelo por sí solo.

---
# 1. Enfoque elegido

## 1.1 La pregunta que venimos a responder (P1 → H2)

La **Hipótesis 2** del Práctico 1 planteaba que es posible predecir la pérdida o creación de empleo de una empresa a partir de sus características de adopción de IA, proponiendo originalmente `ai_use_case` (el tipo de tarea automatizada) como mecanismo central — con el argumento de que *qué* se automatiza importa más que *cuánto*.

El Práctico 2 (§2.5, §2.6) debilitó esa formulación puntual: `ai_use_case` resultó estar determinada por la industria (cada industria sortea entre un menú fijo de 4 casos de uso) y no aporta información propia de la empresa una vez que `industry` ya está en el modelo. Mantenemos la pregunta de fondo de la H2 — *¿la intensidad y la forma de adopción de IA predicen el resultado laboral de la empresa?* — pero la respondemos con el conjunto completo de predictores de adopción, estructura y gobernanza que sí mostraron estructura real (P2, §11.2), en lugar de apoyarnos en una única variable categórica sin señal propia.

## 1.2 Por qué un enfoque mixto: supervisado + no supervisado

La pregunta tiene dos lecturas legítimas, y cada una pide una técnica distinta:

| Lectura de la H2 | Pregunta que responde | Técnica |
|---|---|---|
| ¿Se puede **predecir** el resultado de empleo de una empresa a partir de cómo adoptó la IA? | Predicción puntual | **Supervisado** (clasificación) |
| ¿Existen **perfiles** de empresa adoptadora que se comporten distinto frente al empleo, más allá de lo que capture una sola variable a la vez? | Estructura latente | **No supervisado** (clustering) |

No son alternativas — son complementarias. El enfoque supervisado da una respuesta directa y cuantificable a "se puede predecir, con tal desempeño". El no supervisado no predice nada: agrupa empresas por similitud en su forma de adoptar IA sin mirar nunca el resultado, y solo después compara el resultado promedio entre grupos. Si las dos vías coinciden — los perfiles de adopción más intensiva muestran, en promedio, mejor balance de creación/desplazamiento, y el clasificador logra separar esos mismos casos — la evidencia a favor de la H2 es más sólida que la que daría cualquiera de las dos por separado, porque son dos formas de ataque metodológicamente independientes sobre la misma pregunta.

## 1.3 Enfoque supervisado — dos clasificadores binarios

Descartamos clasificar el signo de `saldo_neto_empleo` por el motivo ya mencionado: es la resta de dos cantidades con fiabilidad alta, y restar cancela señal y suma ruido (P2, registro de decisiones, punto 6). En su lugar entrenamos **dos clasificadores binarios independientes**, cada uno sobre una tasa con fiabilidad alta partida por su mediana:

| Target | Definición | Fiabilidad (P2 §10.6.2) | Corte |
|---|---|---|---|
| `alta_creacion` | 1 si `tasa_creacion` ≥ mediana, 0 si no | 0,90 | Mediana → balance 50/50 por construcción |
| `alto_desplazamiento` | 1 si `tasa_desplazamiento` ≥ mediana, 0 si no | 0,79 | Mediana → balance 50/50 por construcción |

Los predictores en ambos casos son los 25 del catálogo `PREDICTORES` de P2 §11.2 (estructura, adopción, operación, gobernanza, contexto país), excluyendo siempre las variables de rol *resultado* (serían fuga de información/circularidad, no predictores contemporáneos). `task_automation_rate` se incluye pero se reporta también un modelo sin ella, por su cercanía conceptual al mecanismo de desplazamiento.

Partir por la mediana en vez de por cero tiene una ventaja adicional sobre clasificar el saldo neto: el balance de clases queda 50/50 *por construcción*, así que no hace falta ninguna técnica de manejo de desbalance — el punto 2 del práctico (preparación final) se simplifica en ese aspecto.

## 1.4 Enfoque no supervisado — perfiles de adopción (clustering)

> **Regla de separación (P2, §11.2):** si se clusteriza incluyendo variables de resultado, los grupos se separan por resultado — y después "descubrir" que el grupo que más adoptó IA es el que mejor le fue es circular. El clustering se arma solo con predictores; los resultados se miran después, para describir.

Usamos las variables numéricas de los mismos cinco bloques de predictores (`PRED_ESTRUCTURA`, `PRED_ADOPCION`, `PRED_OPERACION`, `PRED_GOBERNANZA`, `PRED_PAIS`) para agrupar a las empresas por similitud, estandarizando antes de calcular distancias. Las categóricas (`industry`, `region`, `ai_adoption_stage`, etc.) quedan afuera del clustering propiamente dicho y se usan, junto con `tasa_creacion` y `tasa_desplazamiento`, para describir cada clúster una vez formado — nunca para formarlo.

La técnica (K-means o alternativa) y el número de clusters se deciden en la sección de implementación, con un criterio cuantitativo (silhouette / codo) combinado con que los grupos resulten interpretables en términos de negocio.

## 1.5 Ventajas del enfoque

- Ataca el problema real de fiabilidad en vez de ignorarlo: modela las dos variables más confiables del dataset en lugar de su resta ruidosa.
- El corte por mediana da balance de clases perfecto, sin necesidad de sobremuestreo/submuestreo.
- Permite un hallazgo más rico que un solo número de accuracy: si los predictores de "alta creación" y "alto desplazamiento" resultan distintos, sugiere que crear y destruir empleo son mecanismos separados, no dos caras de la misma moneda — algo que el saldo neto, al fusionarlos en una resta, no podría mostrar nunca.
- El clustering aporta una segunda fuente de evidencia, metodológicamente independiente del clasificador, para la misma pregunta (triangulación).
- Ninguno de los dos métodos usa variables de rol *resultado* como predictoras, así que ninguno corre el riesgo de un modelo tautológico.

## 1.6 Limitaciones y riesgos identificados

- **Sigue habiendo un techo de fiabilidad**, aunque más alto que con el saldo neto (0,90 y 0,79 en vez de 0,33): parte de la varianza de cada tasa sigue siendo ruido trimestral, así que ningún clasificador va a acercarse a un accuracy perfecto, y no debería esperarse eso.
- **Partir por la mediana es relativo, no absoluto**: una empresa justo por encima y otra justo por debajo de la mediana pueden tener tasas de creación casi idénticas. El corte separa "la mitad superior" de la distribución, no necesariamente dos poblaciones cualitativamente distintas.
- **`task_automation_rate` sigue siendo un caso límite** conceptualmente cercano al mecanismo que se quiere probar; reportar el modelo con y sin ella mitiga el riesgo pero no lo elimina del todo.
- **El clustering no prueba causalidad.** Si un clúster de alta adopción muestra mejor balance de empleo, es una asociación observacional entre perfiles — no evidencia de que la adopción de IA *cause* ese resultado.
- **El dataset es sintético y generado por reglas** (P2, conclusiones finales): cualquier patrón que encuentren los dos métodos puede ser un artefacto del generador y no un fenómeno real del mercado laboral. Esto no invalida el ejercicio, pero limita qué tan fuerte puede ser el lenguaje de las conclusiones.

In [3]:
!git clone -b main --depth 1 https://github.com/Franncippi/Proyecto-mentoria-M09.git repo_m09

Cloning into 'repo_m09'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 26 (delta 0), reused 19 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (26/26), 13.74 MiB | 16.05 MiB/s, done.


In [5]:
import pandas as pd
from pathlib import Path

PROCESSED = Path("repo_m09/data/processed")
empresas = pd.read_parquet(PROCESSED / "empresas.parquet")

print(empresas.shape)              # (10000, 65)
print(empresas.isna().sum().sum())  # 0

(10000, 65)
0
